In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 170)
np.set_printoptions(suppress=True)

In [2]:
import os
from google.colab import files
files.upload()

DATA, BOOK = 'knhanes_like_week4.csv', 'codebook_week4.csv'
df = pd.read_csv(DATA)
w  = df['wt'].to_numpy()
ST, PS = df['kstrata'].to_numpy(), df['psu'].to_numpy()

print(f'n = {len(df):,}  |  조사구 {df.psu.nunique()}개  |  층 {df.kstrata.nunique()}개')
print(f'가중치 합 = {w.sum():,.0f}')

Saving codebook_week4.csv to codebook_week4.csv
Saving knhanes_like_week4.csv to knhanes_like_week4.csv
n = 5,367  |  조사구 263개  |  층 18개
가중치 합 = 450,899


In [3]:
import pandas as pd
import numpy as np

# 1. 데이터 불러오기
df = pd.read_csv('knhanes_like_week4.csv')

# 2. 노출 변수 설정 (팀 6번: 저학력, edu <= 2)
df['exposure'] = (df['edu'] <= 2).astype(int)  # 1 = 노출군, 0 = 비노출군

# 가중 평균 및 표준편차 계산 함수
def w_mean_sd(values, weights):
    valid = ~np.isnan(values)
    v = values[valid]
    w = weights[valid]
    mean = np.average(v, weights=w)
    variance = np.average((v - mean)**2, weights=w)
    sd = np.sqrt(variance)
    return mean, sd

# 가중 백분율(%) 계산 함수
def w_prop(series, weights, target_val):
    valid = ~series.isna()
    s = (series[valid] == target_val).astype(int)
    w = weights[valid]
    prop = np.average(s, weights=w) * 100
    return prop

# 분석할 변수 목록
variables = ['age', 'sex', 'urban', 'income', 'bmi', 'sbp', 'dbp', 'chol', 'glucose', 'smk', 'drink', 'exercise', 'htn']
rows = []

for c in variables:
    if c in ['age', 'bmi', 'sbp', 'dbp', 'chol', 'glucose', 'income']:
        m_tot, sd_tot = w_mean_sd(df[c], df['wt'])
        m_1, sd_1 = w_mean_sd(df[df['exposure']==1][c], df[df['exposure']==1]['wt'])
        m_0, sd_0 = w_mean_sd(df[df['exposure']==0][c], df[df['exposure']==0]['wt'])
        rows.append([c, f"{m_tot:.1f} ({sd_tot:.1f})", f"{m_1:.1f} ({sd_1:.1f})", f"{m_0:.1f} ({sd_0:.1f})"])
    elif c == 'sex': # 여성(2) 비율
        p_tot = w_prop(df[c], df['wt'], 2)
        p_1 = w_prop(df[df['exposure']==1][c], df[df['exposure']==1]['wt'], 2)
        p_0 = w_prop(df[df['exposure']==0][c], df[df['exposure']==0]['wt'], 2)
        rows.append([c, f"{p_tot:.1f}%", f"{p_1:.1f}%", f"{p_0:.1f}%"])
    elif c == 'urban': # 읍면부(2) 비율
        p_tot = w_prop(df[c], df['wt'], 2)
        p_1 = w_prop(df[df['exposure']==1][c], df[df['exposure']==1]['wt'], 2)
        p_0 = w_prop(df[df['exposure']==0][c], df[df['exposure']==0]['wt'], 2)
        rows.append([c, f"{p_tot:.1f}%", f"{p_1:.1f}%", f"{p_0:.1f}%"])
    elif c == 'smk': # 현재흡연(1) 비율
        p_tot = w_prop(df[c], df['wt'], 1)
        p_1 = w_prop(df[df['exposure']==1][c], df[df['exposure']==1]['wt'], 1)
        p_0 = w_prop(df[df['exposure']==0][c], df[df['exposure']==0]['wt'], 1)
        rows.append([c, f"{p_tot:.1f}%", f"{p_1:.1f}%", f"{p_0:.1f}%"])
    elif c == 'drink': # 고위험음주(3) 비율
        p_tot = w_prop(df[c], df['wt'], 3)
        p_1 = w_prop(df[df['exposure']==1][c], df[df['exposure']==1]['wt'], 3)
        p_0 = w_prop(df[df['exposure']==0][c], df[df['exposure']==0]['wt'], 3)
        rows.append([c, f"{p_tot:.1f}%", f"{p_1:.1f}%", f"{p_0:.1f}%"])
    elif c == 'exercise': # 규칙적 운동 '예'(2) 비율
        p_tot = w_prop(df[c], df['wt'], 2)
        p_1 = w_prop(df[df['exposure']==1][c], df[df['exposure']==1]['wt'], 2)
        p_0 = w_prop(df[df['exposure']==0][c], df[df['exposure']==0]['wt'], 2)
        rows.append([c, f"{p_tot:.1f}%", f"{p_1:.1f}%", f"{p_0:.1f}%"])
    elif c == 'htn': # 고혈압 있음(1) 비율
        p_tot = w_prop(df[c], df['wt'], 1)
        p_1 = w_prop(df[df['exposure']==1][c], df[df['exposure']==1]['wt'], 1)
        p_0 = w_prop(df[df['exposure']==0][c], df[df['exposure']==0]['wt'], 1)
        rows.append([c, f"{p_tot:.1f}%", f"{p_1:.1f}%", f"{p_0:.1f}%"])

# 최종 DataFrame 생성 및 출력
table1_df = pd.DataFrame(rows, columns=['변수명', '전체 (Total)', '노출군 (저학력)', '비노출군 (고학력 이상)'])
print(f"전체 대상자 수(N): {len(df)} | 노출군(n): {len(df[df['exposure']==1])} | 비노출군(n): {len(df[df['exposure']==0])}")
display(table1_df)

전체 대상자 수(N): 5367 | 노출군(n): 1761 | 비노출군(n): 3606


,변수명,전체 (Total),노출군 (저학력),비노출군 (고학력 이상)
0,age,49.1 (15.8),60.0 (17.0),45.4 (13.5)
1,sex,51.0%,51.9%,50.6%
2,urban,22.9%,35.8%,18.5%
3,income,2.9 (1.8),2.6 (2.3),2.9 (1.6)
4,bmi,23.8 (3.7),24.1 (3.7),23.7 (3.7)
5,sbp,122.8 (18.0),129.0 (18.6),120.6 (17.2)
6,dbp,73.4 (10.5),75.8 (10.5),72.5 (10.3)
7,chol,170.4 (32.2),172.8 (31.2),169.5 (32.4)
8,glucose,91.6 (17.2),94.5 (17.0),90.6 (17.1)
9,smk,24.1%,24.6%,24.0%


## 각주 작성

1. n은 관측 수, %와 평균은 가중 추정치라는 것
2. 연속형 변수를 어떤 통계량으로 요약했는지
3. 집단 비교에 사용한 방법 (SMD 기준 포함)
4. 결측이 있었다면 그 처리 방식

> **각주:**

1.표에 제시된 n은 결측치를 제외한 단순 관측 수이며 백분율(%)과 평균은 복합표본 가중치, 층화변수, 조사구가 반영된 가중 추정치임.

2.연속형 변수는 가중 평균과 가중 표준오차(SE)로 요약하였음.

3.노출군(저학력)과 비노출군(고학력) 간의 집단 비교는 복합표본 도메인 분석과 표준화 평균 차이(SMD)를 활용하였으며 SMD 절댓값이 0.1 미만인 경우는 무시할 만한 차이로 판정함.

4.분석 과정에서 결측값이 존재하는 변수는 해당 변수의 분석 시 결측을 제외하여 처리함.